# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into *record sets* which are collections of records. Each record set contains *fields*, and each field has an `@id` which uniquely identifies it in the dataset.

Let's inspect available record sets and summarize their fields.

In [ ]:
# Get all record sets using their @id
record_sets_info = dataset.record_sets()
record_sets_ids = list(record_sets_info.keys())

print("Record Set @id list:")
for rs_id in record_sets_ids:
    print(f"- {rs_id}")

# For each record set, print its fields (@id)
for rs_id in record_sets_ids:
    record_set = record_sets_info[rs_id]
    print(f"\nRecord Set: {rs_id}")
    fields = [field['@id'] for field in record_set['fields']]
    print(f"Fields (@id): {fields}")

# Show a preview of records from the first record set
if record_sets_ids:
    rs_id = record_sets_ids[0]
    print(f"\nPreview records from {rs_id}:")
    for rec in dataset.records(record_set=rs_id):
        print(rec)
        break  # Print only one record for preview

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here we extract all data from each record set using their `@id`, and convert them to Pandas DataFrames for easier analysis.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet {record_set_id} fields (@id): {df.columns.tolist()}")
    # Show head of DataFrame
    print(df.head(), "\n")

# For example, display records of the primary record set
primary_record_set_id = record_sets_ids[0] if record_sets_ids else None
if primary_record_set_id:
    print(f"Columns for {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Here, we'll select one numeric field (e.g., GAD-7 score) for demonstration, reference it by its `@id`, filter for high scores, normalize, and group by demographic field such as age group (if available).

In [ ]:
# Choose a numeric field for analysis by @id

# If you don't know the exact @id, uncomment below to list all columns:
# print(dataframes[primary_record_set_id].columns.tolist())

# Example choices based on typical survey fields:
# Replace with actual field @ids as found in section 2.
numeric_field_id = None
for col in dataframes[primary_record_set_id].columns:
    if "GAD7" in col or "PHQ9" in col or "psq" in col:
        numeric_field_id = col
        break

# Fallback: choose any integer/float column
if not numeric_field_id:
    for col in dataframes[primary_record_set_id].columns:
        if dataframes[primary_record_set_id][col].dtype in [int, float]:
            numeric_field_id = col
            break

if numeric_field_id:
    threshold = 10
    filtered_df = dataframes[primary_record_set_id][dataframes[primary_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a possible demographic field
    # Demo: pick group_field by @id, e.g., 'age' or 'level_of_education'
    group_field = None
    for col in dataframes[primary_record_set_id].columns:
        if "age" in col.lower() or "education" in col.lower() or "gender" in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we'll plot the distribution of the numeric field selected above, and if grouped, display group means.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    dataframes[primary_record_set_id][numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'grouped_df' in locals():
        plt.figure(figsize=(8, 5))
        plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using the `mlcroissant` library, we loaded and explored the A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya.
- The dataset contains multiple record sets and fields, all uniquely referenced by their `@id`.
- We visualized distributions of mental health scores and grouped data by demographic fields, demonstrating AI-ready interoperability for further analysis.
- For more advanced tasks, you can extend this notebook with deeper statistical analyses or machine learning workflows, always referencing entities with their `@id` per the Croissant schema.